# Confluence API Client

This notebook imports the reusable client from `confluence_client.py` and reads credentials from `.env`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from confluence_client import ConfluenceAPIError, ConfluenceClient
from jira_client import JiraAPIError, JiraClient

client = ConfluenceClient.from_env()
jira_client = JiraClient.from_env()

In [ ]:
current_user = client.get_current_user()
{
    "display_name": current_user.get("displayName") or current_user.get("publicName"),
    "account_id": current_user.get("accountId"),
    "confluence_url": client.config.base_url,
}

In [ ]:
spaces = client.list_spaces(limit=5)
[
    {
        "key": space.get("key"),
        "name": space.get("name"),
        "type": space.get("type"),
    }
    for space in spaces.get("results", [])
]

In [ ]:
try:
    jira_roles_response = jira_client.list_application_roles()
    jira_apps = [
        {
            "key": app.get("key"),
            "name": app.get("name"),
            "user_count": app.get("userCount"),
            "groups": app.get("groups", []),
        }
        for app in jira_roles_response.get("items", [])
    ]
except JiraAPIError as error:
    jira_apps = [
        {
            "error": str(error),
            "detail": error.response_body[:500],
        }
    ]

try:
    jira_response = jira_client.list_projects(max_results=50)
    jira_projects = [
        {
            "key": project.get("key"),
            "name": project.get("name"),
            "project_type": project.get("projectTypeKey"),
            "url": project.get("self"),
        }
        for project in jira_response.get("values", [])
    ]
except JiraAPIError as error:
    jira_projects = [
        {
            "error": str(error),
            "detail": error.response_body[:500],
        }
    ]

pages_response = client.list_pages(limit=20)
confluence_pages = []
for result in pages_response.get("results", []):
    page = result.get("content", result)
    links = page.get("_links", {})
    space = page.get("space", {})
    version = page.get("version", {})
    confluence_pages.append(
        {
            "title": page.get("title"),
            "space": space.get("name") or space.get("key"),
            "last_updated": version.get("when"),
            "url": f"{client.config.base_url.rstrip('/')}{links.get('webui', '')}",
        }
    )

{
    "jira_apps": jira_apps,
    "jira_projects": jira_projects,
    "confluence_pages": confluence_pages,
}

In [ ]:
page_url="https://cgi-team-agentic.atlassian.net/spaces/MFS/pages/7864321/JIRA+Links"

page_context = client.get_page_context_by_url(page_url)
jira_macros = jira_client.extract_jira_macros_from_page_context(page_context)


In [ ]:
{
    "page_context": page_context,
    "jira_macros": jira_macros,
}

In [ ]:
page_jira_links = jira_client.get_jira_links_from_page_context(page_context)


In [ ]:
page_jira_links